# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: FAIR² Dataset Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library. We focus on step-by-step exploration structured by Croissant schema entities identified via their `@id`, as recommended for reproducible FAIR data workflows.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets and fields by their `@id` values.

In [ ]:
# Get available record sets with their @id
record_sets = dataset.metadata.recordSet
print("Record Sets in the dataset:")
if record_sets:
    for rs in record_sets:
        print(f"  - @id: {rs['@id']} (type: {rs.get('@type', 'RecordSet')})")
else:
    print("No record sets found in metadata. Inspecting distributions.")
    distributions = dataset.metadata.distribution
    for dist in distributions:
        print(f"  - distribution @id: {dist['@id']}")

# Try printing fields/columns for each record set or distribution
if record_sets:
    for rs in record_sets:
        print(f"\nFields for record set @id: {rs['@id']}")
        fields = rs.get('field', [])
        for fld in fields:
            print(f"    - field @id: {fld['@id']}, name: {fld.get('name', '')}, type: {fld.get('@type', '')}")
else:
    for dist in distributions:
        columns = dist.get('column', [])
        if columns:
            print(f"\nColumns for distribution @id: {dist['@id']}")
            for col in columns:
                print(f"    - column @id: {col['@id']}, name: {col.get('name', '')}, type: {col.get('@type', '')}")

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. All entities are referenced by their `@id`.

In [ ]:
# Determine which record sets or distributions to use by their @id
# Since metadata.recordSet appears to be empty, use distribution @ids

distribution_ids = [
    'http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7862866/_/62935be8-24c5-4111-9be6-0b2e3d9593bd',
    'http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7862866/_/53120815-c24d-449d-996b-edc3f2826adc'
]

dataframes = {}
for dist_id in distribution_ids:
    records = list(dataset.records(record_set=dist_id))
    df = pd.DataFrame(records)
    dataframes[dist_id] = df
    print(f"\nLoaded DataFrame for distribution @id: {dist_id}")
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head(2))

# Select first distribution as main dataset for further analysis
primary_dist_id = distribution_ids[0]

## 4. Exploratory Data Analysis (EDA)
Apply exploratory steps such as filtering numeric values, normalization, and grouping, referencing all fields by their `@id`.

In [ ]:
# View all column names to select field @id for numeric and grouping operations
df = dataframes[primary_dist_id]
print("\nAll columns in primary data:")
pprint(df.columns.tolist())

# Example fields (adapt as needed): Assume 'Age' present as personalSensitiveInformation
numeric_field_id = 'Age'   # Use exact field name/column as in schema or DataFrame
group_field_id = 'Sex'     # Another example field for grouping

# Filtering records: Age > 50
threshold = 50
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by Sex (@id)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in DataFrame columns.")

## 5. Visualization
Visualize distributions or relationships of numeric and categorical fields such as Age and Sex. All axes/legends reference fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of Age
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f'Distribution of {numeric_field_id} (@id)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot Age by Sex
if numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} distribution by {group_field_id} (@id)')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook provides an overview and initial exploration of the FAIR² dataset using `mlcroissant`. Following FAIR and Croissant schema best practices, all entities (record sets, fields, columns) were referenced by their `@id` throughout, supporting reproducible and interoperable workflows.

Key findings and steps:
- Loaded metadata and reviewed available distributions and fields via `@id`.
- Extracted tabular records for further analysis and visualization.
- Performed basic EDA: filtered, normalized, and grouped numeric data (e.g., Age) using field `@id`.
- Generated simple visualizations for main clinical variables.

Further analysis can be conducted by referencing additional record sets, fields, or columns via their `@id`, supporting advanced clinical, epidemiological, or biomarker studies.